In [2]:
print("Hello Ai Agents!")

Hello Ai Agents!


In [21]:
from dataclasses import dataclass
from openai import OpenAI
import os
import json

@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests accross all inference providers """

    name: str
    env_var: str
    is_true: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider("Groq","GROQ_API_KEY",True,"https://api.groq.com/openai/v1","qwen/qwen3.8-27b"),
    Provider("OpenRouter","OPENROUTER_API_KEY",True,"https://openrouter.ai/api/v1","stealth/space-bunny-alpha")
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")


def build_client(provider: Provider) -> OpenAI:

    api_key = os.getenv(provider.env_var)


    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )

def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")



def model_reply(prompt: str) -> str :

    provider = select_provider()
    print(f"Using {provider.name} provider and {provider.model} model")
    client = build_client(provider)

    reply = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return reply.choices[0].message.content

Found a provider key.


In [10]:
WEATHER_DB = {
    "london": {"celcius": 22, "sky": "cloudy"},
    "paris": {"celcius": 24, "sky": "sunny"},
    "tallinn": {"celcius": 28, "sky": "sunny"},
    "moscow": {"celcius": 18, "sky": "rainy"},
    "tokyo": {"celcius": 30, "sky": "sunny"},
    "beijing": {"celcius": 26, "sky": "cloudy"},
    "new york": {"celcius": 28, "sky": "sunny"},
    "mumbai": {"celcius": 32, "sky": "sunny"},
    "cape town": {"celcius": 16, "sky": "cloudy"},
    "sydney": {"celcius": 20, "sky": "sunny"},
    "rio de janeiro": {"celcius": 22, "sky": "cloudy"},
    "cairo": {"celcius": 24, "sky": "sunny"},
    "mexico city": {"celcius": 20, "sky": "cloudy"},
    
}

In [11]:
def lookup_weather(location: str) -> str:
    """Look up the weather for a location."""

    record = WEATHER_DB.get(location.lower())
    if record is None:
        return f"No data found for this {location}"

    return f"The weather in {location} is {record['celcius']}C and {record['sky']}."
    


Below is a tool schema defined manually by us, which can be called on our machine, and llm will just signal should we call this tool or not ?
and this schema has all the detail about aur tool like parameters it required and tool name.

In [ ]:
# the schema will be given to the llm
weather_Schema = {
    "type": "function",
    "function": {
        "name": "lookup_weather",
        "description": "This is a look up weather for a location",
        "parameters":{
            "type":"object",
            "properties": {
                "location": {"type": "string", "description": "The location to look up the weather for."}
            },
            "required": ["location"]
        }
    }
}

TOOLS = {
    "lookup_weather": lookup_weather,
    "stock_price_checker": None
}

In [13]:
def ask_llm_with_tool(prompt: str,*, max_tokens: int =400):
    """Calls the llm with tool call"""

    provider = select_provider()
    client = build_client(provider)

    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=max_tokens,
        messages= [
            {
                "role":"user",
                "content": prompt
            }
        ],
        tools=[weather_Schema],
    )

    print(result.choices)

    return result.choices[0].message

In [ ]:
msg = ask_llm_with_tool("What is the weather in moscow compared to tokyo?")

print(f"LLM final response : {msg}")

# Response format which has message.content = None but with a tool call detail
# {
#   "choices": [
#     {
#       "index": 0,
#       "finish_reason": "tool_calls",
#       "logprobs": null,
#       "message": {
#         "role": "assistant",
#         "content": null,
#         "refusal": null,
#         "annotations": null,
#         "audio": null,
#         "function_call": null,
#         "tool_calls": [
#           {
#             "id": "fc_25211c11-9883-4c06-b12f-f3f8b26239c7",
#             "type": "function",
#             "function": {
#               "name": "lookup_weather",
#               "arguments": "{\"location\":\"London\"}"
#             }
#           }
#         ],
#         "reasoning": "User wants weather in London. Need to call lookup_weather."
#       }
#     }
#   ]
# }


[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='qm1pgzptf', function=Function(arguments='{"location":"moscow"}', name='lookup_weather'), type='function'), ChatCompletionMessageFunctionToolCall(id='1a5x2n278', function=Function(arguments='{"location":"tokyo"}', name='lookup_weather'), type='function')]))]
LLM final response : ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='qm1pgzptf', function=Function(arguments='{"location":"moscow"}', name='lookup_weather'), type='function'), ChatCompletionMessageFunctionToolCall(id='1a5x2n278', function=Function(arguments='{"location":"tokyo"}', name='lookup_weather'), type='function')])


In [23]:
if msg.tool_calls:
    print("tool call detected")
    print("Total tool calls", len(msg.tool_calls))
    for tool_call in msg.tool_calls:
        print("Tool call name", tool_call.function.name)
        print("Tool call arguments", tool_call.function.arguments)
        city = json.loads(tool_call.function.arguments)["location"]
        print(f"Calling tool for {city}")
        tool = TOOLS[tool_call.function.name] # this will give us the function 
        result = tool(city)
        print(result)

        print("-"*100)
else:
    print("No tool call detected")

tool call detected
Total tool calls 2
Tool call name lookup_weather
Tool call arguments {"location":"moscow"}
Calling tool for moscow
The weather in moscow is 18C and rainy.
----------------------------------------------------------------------------------------------------
Tool call name lookup_weather
Tool call arguments {"location":"tokyo"}
Calling tool for tokyo
The weather in tokyo is 30C and sunny.
----------------------------------------------------------------------------------------------------


## Execution Loop (Agent Loop)

In [28]:
def ask_llm_with_tool_call(prompt: str,*,max_tokens:int = 400):

    provider = select_provider()
    client = build_client(provider)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    while True:
        msg_result = client.chat.completions.create(
            model=provider.model,
            max_tokens=max_tokens,
            messages=messages,
            tools=[weather_Schema]
        )

        msg = msg_result.choices[0].message
        if not msg.tool_calls:
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": msg.tool_calls
        })

        for tool_call in msg.tool_calls:
            tool_call_name = tool_call.function.name
            tool_call_args = json.loads(tool_call.function.arguments) # {"location":"london"}
            tool_call_id = tool_call.id

            if tool_call_name not in TOOLS:
                raise ValueError(f"Tool {tool_call_name} not found")

            tool = TOOLS[tool_call_name]

            result = tool(**tool_call_args)

            messages.append({
                "role":"tool",
                "tool_call_id": tool_call_id,
                "content": result
            })



In [33]:
llm_response = ask_llm_with_tool_call("What is the weather in london compared to paris?")

print(llm_response)

Here is the weather comparison between London and Paris:

*   **London:** 22°C, Cloudy
*   **Paris:** 24°C, Sunny

Paris is slightly warmer (by 2°C) and enjoying sunny weather, while London is a bit cooler with cloudy skies.
